## Import and format the data to build a DataLoader

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

In [21]:
url =  Path.cwd()/ "iris.txt"
df = pd.read_table(url, sep=",", header=None)

df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [22]:
df.columns = ["sepal_length",
              "sepal_width",
              "petal_length",
              "petal_width",
              "class"]




input_values = df[['petal_width', 'sepal_width']]
label_values = df['class']
class_as_numbers = label_values.factorize()[0]


input_train, input_test, label_train, label_test = train_test_split(input_values,
                                                                    class_as_numbers,
                                                                    test_size=0.25,
                                                                    stratify=class_as_numbers,
                                                                    random_state=42)


one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

max_vals_in_input_train = input_train.max()
min_vals_in_input_train = input_train.min()


#  Next we normalize input_train adn input_test with max and min values from input_train
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)


input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)

train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

## Building the Neural Network

In [27]:
class myNN(L.LightningModule):
    def __init__(self):
        super().__init__()        
        L.seed_everything(seed=42)

        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
        self.loss = nn.CrossEntropyLoss()

    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))
        return output_values

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.001)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)
        return loss


In [28]:
model = myNN()

Seed set to 42


In [29]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name             | Type             | Params | Mode  | FLOPs
----------------------------------------------------------------------
0 | input_to_hidden  | Linear           | 6      | train | 0    
1 | hidden_to_output | Linear           | 9      | train | 0    
2 | loss             | CrossEntropyLoss | 0      | train | 0    
----------------------------------------------------------------------
15        Trainable params
0         Non-trainable params
15        Total params
0.

Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 438.70it/s, v_num=3]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 429.52it/s, v_num=3]


In [ ]:
predictions - model(input_)